In [599]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import mannwhitneyu

from lifelines import KaplanMeierFitter
from lifelines import CoxPHFitter
from lifelines.statistics import logrank_test

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (make_scorer,roc_auc_score,roc_curve, precision_score, fbeta_score, f1_score, recall_score,
confusion_matrix, classification_report, accuracy_score, ConfusionMatrixDisplay)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, RandomizedSearchCV

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Read Heart Failure Dataset

In [601]:
df = pd.read_csv("..\data\heart_failure_clinical_records_dataset.csv")
df.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


We prepare dataset for machine learning modeling

# Define X and y

In [604]:
drop_cols = ["time", "DEATH_EVENT"] # we droped time to avoid data leakage because follow up time is part of the survival outcomes
X = df.drop(columns = drop_cols)
y = df["DEATH_EVENT"]
X.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0


In [605]:
y.head()

0    1
1    1
2    1
3    1
4    1
Name: DEATH_EVENT, dtype: int64

# Split Dataset

In [607]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Logistic Regression Model

In [609]:
# We extract numeric and categorical features
numeric_cols = ["age", "creatinine_phosphokinase", "ejection_fraction", "platelets", "serum_creatinine", "serum_sodium"]
categorical_cols = ["anaemia", "diabetes", "high_blood_pressure", "sex","smoking"]


In [610]:
#We preprocess the columns in a transformer
preprocessor = ColumnTransformer(transformers =[("num", StandardScaler(), numeric_cols),("cat", "passthrough", categorical_cols)], remainder="drop")

In [611]:
# We buid a pipeline
log_pipeline = Pipeline(steps=[("preprocessor", preprocessor),
                        ("classifier",LogisticRegression(max_iter=1000, solver="lbfgs", C = 0.1, class_weight="balanced", random_state= 42))])

In [612]:
# We fit the model
log_pipeline.fit(X_train, y_train);

We check the model performance on training set

In [614]:
y_train_pred = log_pipeline.predict(X_train)
print(classification_report(y_train, y_train_pred, target_names=["no_death_event","death_event"]))

                precision    recall  f1-score   support

no_death_event       0.88      0.78      0.83       162
   death_event       0.63      0.77      0.69        77

      accuracy                           0.78       239
     macro avg       0.75      0.78      0.76       239
  weighted avg       0.80      0.78      0.78       239



We check the model performance on test set

In [616]:
y_test_pred = log_pipeline.predict(X_test)
print(classification_report(y_test, y_test_pred, target_names=["no_death_event", "death_event"]))

                precision    recall  f1-score   support

no_death_event       0.84      0.78      0.81        41
   death_event       0.59      0.68      0.63        19

      accuracy                           0.75        60
     macro avg       0.72      0.73      0.72        60
  weighted avg       0.76      0.75      0.75        60



In [617]:
cm = confusion_matrix(y_test, y_test_pred)
print(cm)

[[32  9]
 [ 6 13]]


The logistic regression model correctly identified 68% of the actual death-event cases in the test set. There were 19 actual death-event cases, and the model detected approximately 13 of them. The precision for the death-event class was also 0.59, meaning that among patients predicted as death-event cases, about 59% truly experienced a death event. Although the model shows moderate ability to detect mortality cases, it still misses some of the actual death-event patients, which is an important limitation in a healthcare risk prediction setting.

## Hyperparameter Tuning

We tune the hyperparameters and employ a cross validation for possible improve performance.

In [621]:
para_grid = {"classifier__C": [0.01, 0.02, 0.03,0.04,0.05], "classifier__penalty": ["l2"], 
             "classifier__solver": ["lbfgs"], "classifier__class_weight": ["balanced"] }

In [622]:
cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state =42)
score = make_scorer(recall_score,labels =[1], average="macro")
random_search = RandomizedSearchCV(estimator=log_pipeline,
                          param_distributions= para_grid,
                          n_iter = 100,
                          cv = cv,
                          scoring=score,
                          refit = True,
                          random_state=42,
                          n_jobs=-1)

In [623]:
random_search.fit(X_train, y_train)

C:\Users\kukal\anaconda3\Anaconda_New\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 5 is smaller than n_iter=100. Running 5 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
C:\Users\kukal\anaconda3\Anaconda_New\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'classifier__C': [0.01, 0.02, ...], 'classifier__class_weight': ['balanced'], 'classifier__penalty': ['l2'], 'classifier__solver': ['lbfgs']}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",100
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",make_scorer(r...average=macro)
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation 

In [624]:
print("Best parameters:", random_search.best_params_)

Best parameters: {'classifier__solver': 'lbfgs', 'classifier__penalty': 'l2', 'classifier__class_weight': 'balanced', 'classifier__C': 0.03}


In [625]:
# We choose the best model
best_logistic_model = random_search.best_estimator_

In [626]:
# We predict using the best model
y_train_pred = best_logistic_model.predict(X_train)

In [654]:
# We check performance on training set
print(classification_report(y_train, y_train_pred, target_names=["no_death_event", "death_event"]))

                precision    recall  f1-score   support

no_death_event       0.87      0.80      0.83       162
   death_event       0.63      0.74      0.68        77

      accuracy                           0.78       239
     macro avg       0.75      0.77      0.76       239
  weighted avg       0.79      0.78      0.78       239



In [628]:
# We check performance on test set
y_test_pred = best_logistic_model.predict(X_test)

In [652]:
# We check the performance
print(classification_report(y_test, y_test_pred, target_names = ["no_death_event", "death_event"]))

                precision    recall  f1-score   support

no_death_event       0.86      0.78      0.82        41
   death_event       0.61      0.74      0.67        19

      accuracy                           0.77        60
     macro avg       0.74      0.76      0.74        60
  weighted avg       0.78      0.77      0.77        60



After applying cross-validation and hyperparameter tuning, the logistic regression model achieved a recall of 0.74 for the death-event class on the test set. This means the model correctly detected approximately 74% of the patients who actually experienced a death event. The model also achieved a precision of 0.61 for the death-event class, meaning that among patients predicted as death-event cases, 61% were truly death-event patients. Overall, the tuned model shows moderate ability to identify mortality-risk cases, with improved sensitivity to death events, although some false positives and false negatives remain. The similarity between the training and test performance suggests that the model generalizes reasonably well and does not show strong evidence of overfitting.